In [1]:
# pip install langchain_huggingface
# !pip install langchain_community

In [2]:
# type(chunks[0])

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
# import spacy
# import faiss

c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16736\840706077.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


load the document

In [4]:
data = open(r'D:\qsp GenAI\RAG\machine_learning_2000_sentences.txt').read()
# data

text normalization

convert all characters to lower

In [5]:
data = data.lower()

remove extra space

In [6]:
import re
data = re.sub(r'\s{2,}', ' ', data)

remove 'machine learning statement 1:' pattern

In [7]:
data = re.sub('machine learning statement \d+\:', '', data)

<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_16736\2448674994.py:1: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  data = re.sub('machine learning statement \d+\:', '', data)


In [8]:
# data

expand contraction and abbreviation

In [9]:
import contractions

In [10]:
data = contractions.fix(data)

punctutations

In [11]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [12]:
data = re.sub(r'[^0-9a-zA-Z\s]','', data)

correct spellings using textblob

In [13]:
from textblob import TextBlob

In [14]:
# str(TextBlob(data).correct())

spacy lemmatization

In [15]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [16]:
tokens = nlp(data)
updated_tokens = [token.lemma_ for token in tokens if not token.is_stop]
# updated_tokens
# if we want to check the word belong to which entity, .ent

In [17]:
data = ' '.join(updated_tokens).strip() # strip to remove extra space, we can also remove characters

chunking

In [18]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 200, chunk_overlap = 40)

In [19]:
# chunks = splitter.split_text(data)
# chunks = list(set(splitter.split_text(data)))
chunks = splitter.create_documents([data])

In [20]:
print(chunks[0])

page_content='workflow emphasizing unsupervise 
 learning improve carefully validate feature scale 
 monitor logistic regression document assumption compare 
 result meaningful baseline deployment machine'


In [21]:
type(chunks[0])

langchain_core.documents.base.Document

In [22]:
print(chunks[0].page_content)

workflow emphasizing unsupervise 
 learning improve carefully validate feature scale 
 monitor logistic regression document assumption compare 
 result meaningful baseline deployment machine


In [ ]:
chunks[0].metadata='data.txt'
# chunks

[Document(metadata='data.txt', page_content='workflow emphasizing unsupervise \n learning improve carefully validate feature scale \n monitor logistic regression document assumption compare \n result meaningful baseline deployment machine'),
 Document(metadata={}, page_content='learn statement 2 workflow emphasize reinforcement learning \n improve carefully validate linear regression monitor \n dimensionality reduction documenting assumption compare result'),
 Document(metadata={}, page_content='meaningful baseline deployment machine learn \n statement 3 workflow emphasize classification improve \n carefully validate neural network monitor randomizedsearchcv'),
 Document(metadata={}, page_content='documenting assumption compare result meaningful \n baseline deployment   workflow \n emphasize regression improve carefully validate tsne \n monitoring reinforcement learning document assumption'),
 Document(metadata={}, page_content='compare result meaningful baseline deployment \n  workflo

In [ ]:
chunks[0].metadata={'file_name':'data.txt'}
# chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='workflow emphasizing unsupervise \n learning improve carefully validate feature scale \n monitor logistic regression document assumption compare \n result meaningful baseline deployment machine'),
 Document(metadata={}, page_content='learn statement 2 workflow emphasize reinforcement learning \n improve carefully validate linear regression monitor \n dimensionality reduction documenting assumption compare result'),
 Document(metadata={}, page_content='meaningful baseline deployment machine learn \n statement 3 workflow emphasize classification improve \n carefully validate neural network monitor randomizedsearchcv'),
 Document(metadata={}, page_content='documenting assumption compare result meaningful \n baseline deployment   workflow \n emphasize regression improve carefully validate tsne \n monitoring reinforcement learning document assumption'),
 Document(metadata={}, page_content='compare result meaningful baseline deploym

### Chunk Embeddings (Converts chunks to vectors)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# model_name = "sentence-transformers/all-mpnet-base-v2"  by default it uses this model if not mentioned any
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-miniLM-L6-V2"
)
# llm_model = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash", # 3.5 flash 2.0 flash, 2.5-pro, 3.6-flash
#     # api_key=os.environ["GEMINI_API_KEY"]

# )

NameError: name 'ChatGoogleGenerativeAI' is not defined

In [ ]:
# response = llm_model.invoke("Explain GenAI?")

In [ ]:
# chunks

In [ ]:
vector_db = FAISS.from_documents(
    documents = chunks, # where we store all documents
    embedding = embedding_model
)
vector_db

In [ ]:
user_query = 'What is Machine Learning'
r_chunks = vector_db.similarity_search(user_query) # returns directly content not distance and index

In [ ]:
# r_chunks = set()
for chunk in r_chunks:
    print(chunk.page_content)

result meaningful baseline deployment machine 
 learning statement 1102 workflow emphasize reinforcement learning 
 improve carefully validate linear regression monitor
result meaningful baseline deployment machine 
 learning statement 1452 workflow emphasize reinforcement learning 
 improve carefully validate linear regression monitor
result meaningful baseline deployment machine 
 learning statement 1052 workflow emphasize reinforcement learning 
 improve carefully validate linear regression monitor
result meaningful baseline deployment machine 
 learning statement 1952 workflow emphasize reinforcement learning 
 improve carefully validate linear regression monitor


Retrieval

In [ ]:
updated_r_chunks = set()
for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)
# updated_r_chunks
R_text = '\n'.join(updated_r_chunks) # it accepts iterables

In [ ]:
R_text

'result meaningful baseline deployment machine \n learning statement 1102 workflow emphasize reinforcement learning \n improve carefully validate linear regression monitor\nresult meaningful baseline deployment machine \n learning statement 1452 workflow emphasize reinforcement learning \n improve carefully validate linear regression monitor\nresult meaningful baseline deployment machine \n learning statement 1052 workflow emphasize reinforcement learning \n improve carefully validate linear regression monitor\nresult meaningful baseline deployment machine \n learning statement 1952 workflow emphasize reinforcement learning \n improve carefully validate linear regression monitor'

if we use llms using langchain, we can use pipeline also
- by using langchain we can create ai agents also.
- we use langchain and langraph to create ai agent
- we won't use huggingface, because we will reach limit. in claude also
- we will download llm models.


Structure the output

In [ ]:
# create gemini api key

In [ ]:
# pip show langchain

Name: langchain
Version: 1.2.12
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# dimension = chunk_embeddings.shape[1]
# dimension

384

Normalize

In [ ]:
# faiss.normalize_L2(chunk_embeddings)

In [ ]:
# index_faiss_db = faiss.IndexFlatIP(dimension)
# index_faiss_db.add(chunk_embeddings)

In [ ]:
# def r_search(query,k=3):
#     query_embeddings = embedding_model.encode(query).astype('float32')
#     query_embeddings = query_embeddings.reshape(1,-1)
#     faiss.normalize_L2(query_embeddings)
#     print(query_embeddings.shape)
#     distance,index = index_faiss_db.search(query_embeddings,k=k)
#     R_chunks = [chunks[i] for i in index[0]]
#     R_str = ' '.join(R_chunks)
#     for chunk in R_chunks:
#         print(chunk.page_content)
#     return R_str


NameError: name 'R_chunks' is not defined

In [ ]:
# def r_search(query,k=3):
#     query_embeddings = embedding_model.encode(query).astype('float32')
#     query_embeddings = query_embeddings.reshape(1,-1)
#     faiss.normalize_L2(query_embeddings)
#     print(query_embeddings.shape)
#     distance,index = index_faiss_db.search(query_embeddings,k=k)
#     R_chunks = [chunks[i] for i in index[0]]
#     R_str = ' '.join(R_chunks)
#     return R_str

# def g_text(r_search):
#         import os
#         import requests
    
#         API_URL = "https://router.huggingface.co/v1/chat/completions"
    
#         headers = {
#             "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
#         }
#         def query(payload):
#             response = requests.post(API_URL, headers=headers, json=payload)
#             return response.json()
#         prompt = f'''
#                     You're an helpful assistant
#                     Assigned Task for you : Structure my output => {r_search}
#                     Note : 
#                     1) Don't add extra contents just structure mentioned output.
#                     2) If there is mistake in output correct or else keep the original output
#                     with structured result.
#             '''
#         response = query({
#             "messages": [
#                 {
#                     "role": "user",
#                     "content": f'{prompt}'  #JSONDecodeError
#                 }
#             ],
#             "model": "deepseek-ai/DeepSeek-R1:novita"
#         })
    
#         return response

# user_prompt = 'Explain Machine Learning ?'
# user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

# r_response = r_search(user_prompt)
# g_response = g_text(r_response)
# print(g_response)

In [ ]:
# change to hugging face

In [ ]:
def r_search(query, k=2): 
    # faiss.normalize_L2(query)
    # dimension = chunk_embeddings.shape[1]
    # index = faiss.IndexFlatIP(dimension)
    query_embedding = embedding_model.encode(query).astype('float32')
    query_embedding = query_embedding.reshape(1,-1)
    faiss.normalize_L2(query_embedding)
    distance, index = index_faiss_db.search(query_embedding,k=k)
    R_chunks = [chunks[i] for i in index[0]]
    R_str = ' '.join(R_chunks)
    return R_str
    # print(query_embedding.shape)
    # for i in index[0]:
    #     print(chunks[i])
    # R_chunks = [chunks[i] for i in index[0]]
    # R_str = ' '.join(R_chunks)
    # return R_str


def g_text(r_search):
    # write the logic of llm
    import os
    import requests
    from openai import OpenAI

    API_URL = "https://router.huggingface.co/v1/chat/completions"

    headers = {
        'Authorization': f"Bearer {os.environ['HF_TOKEN']}",
    }
    
    # client = OpenAI(
    #     base_url="https://router.huggingface.co/v1",
    #     api_key=os.environ["HF_TOKEN"],
    # )

    def query(payload):
        response = requests.post(API_URL, 
                                 headers=headers, 
                                 json=payload,
                                 timeout=90)
        
        print("Status:", response.status_code)
        print("Content-Type:", response.headers.get("Content-Type"))
        print(response.text)
        # return response.json()

    prompt = f'''
            You're an helpful assistant
            Assigned Task for you:
            Structure my output => {r_search}
    '''

    
    response = query({
        "messages" : [
            {
                "role": "user",
                "content": f"{prompt}"
            }
        ],
        "model" : "deepseek-ai/DeepSeek-R1:novita"
        
    })
    return response   
    # print()
    # print(response['choices'][0]['message']['content'])
    # print()



    # prompt = f'''
    #     You're an helpful assistant
    #     Assigned Task for you: Structure my output => {R_str}
    #     Note: 
    #     1.) Don't add extra contents, just structure mentioned output
    #     2.) If there is mistake in output correct or else keep the original output with structured result.
    # '''





# def g_text(r_search):
#     import os
#     import requests
#     from openai import OpenAI
    
#     client = OpenAI(
#         base_url="https://router.huggingface.co/v1",
#         api_key=os.environ["HF_TOKEN"],
#     )
    
#     completion = client.chat.completions.create(
#         model="deepseek-ai/DeepSeek-R1:novita",
#         messages=[
#             {
#                 "role": "user",
#                 "content": prompt
#             }
#         ],
#     )
#     print(completion.choices[0].message)


    # distance, indices = index.search(query_embedding, k=k)
    # return distance, indices

user_prompt = 'Explain Machine Learning?'
user_prompt = re.sub(r'[^0-9a-z-A-Z\s]',' ', user_prompt)

r_response = r_search(user_prompt)
g_response = g_text(r_response)
print(g_response)
print()
print(len(r_response))
print(r_response[:500])

Status: 200
Content-Type: application/json
{"id":"0f580d88f41be7c32e60f7182fc93939","object":"chat.completion","created":1785503490,"model":"deepseek/deepseek-r1-turbo","choices":[{"index":0,"message":{"role":"assistant","content":"<think>\nWe are given a task that appears to be about structuring an output for learning statements (1080 and 1230) that focus on outlier detection workflows. The task also includes instructions to improve, carefully validate, use decision trees, monitor Bayesian optimization, document assumptions, and compare results.\n\nLet's break down the task:\n\n1. **Structure Output**: We need to present the workflow for two learning statements (1080 and 1230) with an emphasis on outlier detection.\n2. **Improve**: We are to improve the workflow.\n3. **Carefully Validate**: Validation is a key step.\n4. **Decision Tree**: Use decision trees in the workflow.\n5. **Monitor Bayesian Optimization**: Incorporate Bayesian optimization and monitor it.\n6. **Document Assumpti

In [ ]:
# print(response['choices'][0]['message']['content'])

# 31 JULY 2026

Another method